# Blocks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/01_blocks.ipynb)

Official API intro to `minilink.blocks`: sources, routing, transfer functions,
nonlinearities, and filters — the vocabulary for wiring diagrams.

**Scripts for depth:** `examples/scripts/blocks/`

**See also:** [`intro/00_core.ipynb`](00_core.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## Sources

`Step`, `WhiteNoise`, and trajectory sources drive open-loop experiments.


In [ ]:
import numpy as np
from minilink.blocks.sources import Step, WhiteNoise

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 2.0
step.show_signal(t0=0.0, tf=5.0)

noise = WhiteNoise(1)
noise.params["t0"] = 0.0
noise.params["tf"] = 5.0
noise.params["sample_period"] = 0.05
noise.params["mean"] = 0.0
noise.params["var"] = 0.25
noise.params["seed"] = 0
noise.show_signal(t0=0.0, tf=5.0)


## Routing and gains

`Gain`, `Sum`, `Mux`, and `Demux` reshape signals inside a diagram.


In [ ]:
import numpy as np
from minilink.blocks.routing import Gain, Sum
from minilink.blocks.sources import Step
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

plant = Pendulum()
plant.x0 = np.array([0.3, 0.0])
step = Step(initial_value=0.0, final_value=0.0, step_time=1.0)
gain = Gain(2.0, dim=1)
loop = (step >> gain) >> plant
loop.compute_trajectory(tf=4.0)
loop.plot_trajectory()
print("Sum block:", Sum)


## Transfer functions and nonlinearities

`TransferFunction` builds LTI blocks from coefficients. Saturation, dead-zone, and
relay live under `minilink.blocks.nonlinear`; filters under `minilink.blocks.filters`.


In [ ]:
from minilink.blocks.transfer_function import TransferFunction
from minilink.blocks.nonlinear import Saturation

# Second-order TF: wn=2, zeta=0.3  ->  wn^2 / (s^2 + 2 zeta wn s + wn^2)
wn, zeta = 2.0, 0.3
tf = TransferFunction([wn**2], [1.0, 2 * zeta * wn, wn**2])
print(tf)

sat = Saturation(lower=-1.0, upper=1.0)
print(sat)
print("More demos: examples/scripts/blocks/")
